# SSPS - UCC Students Success Prediction System
## CGPA prediction model (Python / scikit-learn)

**Author:** Final year project, University of Cape Coast - College of Distance Education (CoDE)

This notebook loads the student dataset, engineers the features, trains and compares
**Ridge Regression** and **Random Forest Regression**, evaluates them with **MAE, RMSE and R^2**,
and exports the coefficients of the deployed model so the web application can score students in real time.

Run it in Jupyter Notebook / JupyterLab (Anaconda).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded")

## 1. Load the dataset\n\nPlace `students_dataset.csv` next to this notebook. Each row is one CoDE student.

In [ ]:
df = pd.read_csv("students_dataset.csv")
print(df.shape)
df.head()

## 2. Features and target

In [ ]:
FEATURES = [
    "level100_gpa", "level200_gpa", "course_credits", "attendance_pct",
    "study_hours_per_week", "participation_score", "quiz1_score", "quiz2_score",
    "assignment_score", "presentation_score", "practical_score",
]
TARGET = "final_cgpa"

X = df[FEATURES]
y = df[TARGET]
df[FEATURES + [TARGET]].describe().T

### Exploratory plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(y, bins=25, color="#1f3c88", edgecolor="white")
ax[0].set_title("Distribution of final CGPA (4.00 scale)")
ax[0].set_xlabel("CGPA")
ax[1].scatter(df["level200_gpa"], y, s=8, alpha=0.5, color="#1f3c88")
ax[1].set_title("Level 200 GPA vs final CGPA")
ax[1].set_xlabel("Level 200 GPA"); ax[1].set_ylabel("Final CGPA")
plt.tight_layout(); plt.show()

corr = df[FEATURES + [TARGET]].corr()[TARGET].sort_values(ascending=False)
corr

## 3. Train / test split (80 / 20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(len(X_train), "training rows |", len(X_test), "test rows")

## 4. Models

Missing continuous-assessment values are median-imputed, then standardised, because
students may leave optional fields blank in the web application.

In [ ]:
ridge = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0, random_state=None)),
])

forest = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1)),
])

ridge.fit(X_train, y_train)
forest.fit(X_train, y_train)
print("Both models trained")

## 5. Evaluation - MAE, RMSE and R^2 on the held-out test set

In [ ]:
def evaluate(name, model):
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    r2 = r2_score(y_test, pred)
    return {"Model": name, "MAE": round(mae, 3), "RMSE": round(rmse, 3), "R2": round(r2, 3)}

results = pd.DataFrame([evaluate("Ridge Regression", ridge),
                        evaluate("Random Forest", forest)])
results

In [ ]:
cv = cross_val_score(ridge, X, y, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="r2")
print("Ridge 5-fold CV R^2: %.3f (+/- %.3f)" % (cv.mean(), cv.std()))

### Evaluation charts

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
results.plot(x="Model", y=["MAE", "RMSE"], kind="bar", ax=ax[0], rot=0,
             color=["#1f3c88", "#f0b429"])
ax[0].set_title("Error comparison (lower is better)")
results.plot(x="Model", y="R2", kind="bar", ax=ax[1], rot=0, color="#1f3c88", legend=False)
ax[1].set_ylim(0, 1); ax[1].set_title("Explained variance R^2 (higher is better)")
plt.tight_layout(); plt.show()

pred = ridge.predict(X_test)
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, s=12, alpha=0.6, color="#1f3c88")
plt.plot([0, 4], [0, 4], "--", color="#c0392b")
plt.xlabel("Actual CGPA"); plt.ylabel("Predicted CGPA")
plt.title("Ridge Regression: predicted vs actual")
plt.tight_layout(); plt.show()

## 6. Feature weights of the deployed Ridge model

In [ ]:
coefs = pd.Series(ridge.named_steps["model"].coef_, index=FEATURES).sort_values()
coefs.plot(kind="barh", figsize=(7, 5), color="#1f3c88")
plt.title("Ridge coefficients (standardised features)")
plt.tight_layout(); plt.show()
coefs.sort_values(ascending=False)

## 7. Export the trained model for the web application

The SSPS web front end scores students instantly in the browser, so the trained
coefficients are exported to JSON and embedded in `src/lib/model.ts`.

In [ ]:
import json

scaler = ridge.named_steps["scale"]
imputer = ridge.named_steps["impute"]
model = ridge.named_steps["model"]

export = {
    "model_version": "ridge-v1",
    "features": FEATURES,
    "means": dict(zip(FEATURES, scaler.mean_.round(6).tolist())),
    "scales": dict(zip(FEATURES, scaler.scale_.round(6).tolist())),
    "medians": dict(zip(FEATURES, np.round(imputer.statistics_, 6).tolist())),
    "coefficients": dict(zip(FEATURES, model.coef_.round(6).tolist())),
    "intercept": round(float(model.intercept_), 6),
    "metrics": evaluate("Ridge Regression", ridge),
    "training_rows": int(len(X_train)),
}
with open("ssps_model.json", "w") as f:
    json.dump(export, f, indent=2)
print(json.dumps(export["metrics"], indent=2))

In [ ]:
import joblib
joblib.dump(ridge, "ssps_ridge_model.joblib")
print("Saved ssps_ridge_model.joblib")

## 8. Continuous learning

Students record their real final CGPA in SSPS after graduation. Export the
`predictions` and `actual_outcomes` tables to CSV from the staff console, append them
to `students_dataset.csv`, and re-run this notebook to retrain and recalibrate the model.